# Gloss-Free Sign Language Translation (ASL-to-English)
## Portfolio Training Notebook for Kaggle GPUs

This notebook provides a complete pipeline to clone the landmark-based ASL translation repository, set up the dependency environment, validate MediaPipe Holistic datasets, run sanitization unit tests, train our hybrid **Conformer-T5 model** with **Multi-Stream Gated Fusion** (fusing 3D landmarks and precomputed I3D features), and export the trained model to ONNX for fast inference.

## 1. Setup, Environment Validation, and Sanity Checks

In [ ]:
# Clone or update the repository
import os

if not os.path.exists("/kaggle/working/gloss-free-asl-translation"):
    !git clone https://github.com/yyouretoast/gloss-free-asl-translation.git
    %cd /kaggle/working/gloss-free-asl-translation
else:
    %cd /kaggle/working/gloss-free-asl-translation
    !git checkout -- requirements.txt
    !git pull

In [ ]:
# Filter out torch/torchvision to keep Kaggle's GPU-optimized pre-installs
!sed -i '/torch/d' requirements.txt
!pip install -r requirements.txt
!pip install -e .
!pip install -q "protobuf>=5.20.0,<6.0.0" --upgrade

In [ ]:
# Run environment validation to verify GPU availability and dependencies
!python -m scripts.check_environment

## 2. Run Sanity Unit Tests

In [ ]:
# Run unit tests (excluding slow dataset integration tests)
!pytest tests/ -v -m "not slow"

## 2. Environment Configuration (Optional HF Token)

In [ ]:
import os
# Set your HF Token if available
# os.environ["HF_TOKEN"] = "your_huggingface_token_here"

## 3. Real-World Dataset Profiling & Validation

In [ ]:
import os
import glob

# Dynamically resolve How2Sign Holistic dataset path on Kaggle
metadata_file = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "realigned_train" in f and f.endswith(".csv"):
            metadata_file = os.path.join(root, f)
            break
    if metadata_file:
        break

data_dir = None
if metadata_file:
    # Landmarks dataset root is the grandparent of realigned_train.csv (sibling to train/val folders)
    data_dir = os.path.dirname(os.path.dirname(metadata_file))

if data_dir and os.path.exists(data_dir):
    print(f"Target data directory resolved: {data_dir}")
    # Run verification limit to verify landmarks are correct
    !python -m src.validate_dataset --data_dir {data_dir} --limit 50
else:
    print(
        "How2Sign Holistic dataset not found! Please attach 'how2sign-holistic' to your Kaggle notebook."
    )

In [ ]:
# Inspect the shapes and channels of the attached Holistic landmarks and I3D features
import os
import numpy as np

# 1. Dynamically resolve paths
metadata_file = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "realigned_train" in f and f.endswith(".csv"):
            metadata_file = os.path.join(root, f)
            break
    if metadata_file:
        break

lm_dir = None
if metadata_file:
    lm_dir = os.path.dirname(os.path.dirname(metadata_file))

i3d_train_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if (
        "i3d" in root.lower()
        and "val" not in root.lower()
        and any(f.endswith(".npy") for f in files)
    ):
        i3d_train_dir = root
        break

# 2. Audit Landmarks
if lm_dir and os.path.exists(lm_dir):
    # Find any folder inside lm_dir that contains .npy files, e.g. train/frontal
    train_frontal_dir = os.path.join(lm_dir, "train", "frontal")
    if not os.path.exists(train_frontal_dir):
        # Fallback to recursively walking lm_dir
        for root, dirs, files in os.walk(lm_dir):
            if any(f.endswith(".npy") for f in files):
                train_frontal_dir = root
                break

    if os.path.exists(train_frontal_dir):
        npy_files = sorted(
            [
                os.path.join(train_frontal_dir, f)
                for f in os.listdir(train_frontal_dir)
                if f.endswith(".npy")
            ]
        )
        if npy_files:
            print(f"Found {len(npy_files)} landmark files in {train_frontal_dir}.")
            test_file = npy_files[0]
            print(f"Auditing landmarks file '{os.path.basename(test_file)}':")
            data = np.load(test_file)
            print(f" - Array Shape: {data.shape} (Expected: (num_frames, 543, 3))")
else:
    print("Landmark Directory not resolved.")

# 3. Audit I3D Features
if i3d_train_dir and os.path.exists(i3d_train_dir):
    i3d_files = sorted(
        [
            os.path.join(i3d_train_dir, f)
            for f in os.listdir(i3d_train_dir)
            if f.endswith(".npy")
        ]
    )
    if i3d_files:
        print(f"Found {len(i3d_files)} I3D feature files in {i3d_train_dir}.")
        test_i3d = i3d_files[0]
        print(f"Auditing I3D file '{os.path.basename(test_i3d)}':")
        data_i3d = np.load(test_i3d)
        print(f" - Feature Shape: {data_i3d.shape} (Expected: (num_frames, 1024))")
else:
    print("I3D Feature Directory not resolved.")

### 4.2 Persist Preprocessed Dataset for Future Runs (Optional)

Since preprocessing the 31,000+ folders of How2Sign takes approximately 2 hours, zipping the preprocessed `.npz` files and saving them to Kaggle's `/kaggle/working` directory allows you to download them or persist them as a Kaggle output dataset. This lets you skip the preprocessing phase entirely in future runs.

## 4. End-to-End Conformer-T5 Multi-Stream Model Training

This runs our end-to-end `Conformer -> T5-Small` translation pipeline using **Multi-Stream Gated Fusion** to combine skeletal trajectories and precomputed I3D spatiotemporal visual features. Adjust batch size and epochs as needed.

In [ ]:
# --- Option A: Load TensorBoard visualization ---
%load_ext tensorboard
%tensorboard --logdir results/checkpoints/runs

In [ ]:
# --- Option A: How2Sign Multi-Stream Training (3 Datasets Dynamic Resolution) ---
import os

# 1. Resolve Metadata CSV file (train split)
metadata_file = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "realigned_train" in f and f.endswith(".csv"):
            metadata_file = os.path.join(root, f)
            break
    if metadata_file:
        break

# 2. Resolve Landmark Directory
lm_dir = None
if metadata_file:
    lm_dir = os.path.dirname(os.path.dirname(metadata_file))

# 3. Resolve Training I3D Directory (parent dataset containing all 10 zip parts)
i3d_train_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "how2sign-i3d-features" in root.lower():
        i3d_train_dir = root
        break

# 4. Resolve Validation I3D Directory (parent dataset containing validation features)
i3d_val_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "how2sign-i3d-val" in root.lower():
        i3d_val_dir = root
        break

print(f"Landmark Directory: {lm_dir}")
print(f"I3D Train Dir:      {i3d_train_dir}")
print(f"I3D Val Dir:        {i3d_val_dir}")
print(f"Metadata File:      {metadata_file}")

if lm_dir and i3d_train_dir and i3d_val_dir and metadata_file:
    # Run training with I3D multi-stream fusion enabled across train & validation splits
    !python -m src.train --epochs 50 --batch_size 8 --lr 1e-4 --data_dir {lm_dir} --i3d_dir {i3d_train_dir} --i3d_val_dir {i3d_val_dir} --metadata_file {metadata_file} --resume_from_checkpoint latest
else:
    print(
        "Error: Missing directories. Please verify how2sign-holistic, how2sign-i3d-features, and how2sign-i3d-val are attached."
    )

## 5. Model Optimization & Export (Conformer Encoder to ONNX)

In [ ]:
# Trace and export the best checkpoint to ONNX
import os

checkpoint_dirs = sorted(
    glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1])
)
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]
    model_bin = os.path.join(best_checkpoint, "model.safetensors")
    if not os.path.exists(model_bin):
        model_bin = os.path.join(best_checkpoint, "pytorch_model.bin")

    # 501 dimensions for face-enabled How2Sign Holistic (.npy), 225 for face-disabled ablation run
    input_dim = 501  # CHANGE this value to 225 if you ran training with --no_face!

    print(f"Exporting encoder from {model_bin} to ONNX with input_dim={input_dim}...")
    !python -X utf8 scripts/export_onnx.py --input-dim {input_dim} --model-path {model_bin} --output /kaggle/working/conformer_encoder.onnx
else:
    print("No checkpoints found to export!")

## 6. Checkpoint Archival & Retrieval

In [ ]:
# Zip all checkpoints for easy download
!zip -q -r /kaggle/working/checkpoints.zip results/checkpoints
print("Zipped checkpoints to /kaggle/working/checkpoints.zip")

## 7. Quantitative Evaluation

In [ ]:
# --- Run Quantitative Evaluation ---
import glob
import os

checkpoint_dirs = sorted(
    glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1])
)
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]

    # Resolve directories dynamically (same as training cell)
    metadata_file = None
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if "realigned_train" in f and f.endswith(".csv"):
                metadata_file = os.path.join(root, f)
                break
        if metadata_file:
            break

    lm_dir = None
    if metadata_file:
        lm_dir = os.path.dirname(os.path.dirname(metadata_file))

    # Resolve Validation I3D Directory
    i3d_val_dir = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if "how2sign-i3d-val" in root.lower() and any(
            f.endswith(".npy") for f in files
        ):
            i3d_val_dir = root
            break

    !pip install -q jiwer sacrebleu

    print(f"Evaluating best checkpoint: {best_checkpoint}")
    print(f" - Landmarks: {lm_dir}")
    print(f" - I3D Val:   {i3d_val_dir}")
    print(f" - Metadata:  {metadata_file}")

    # Pass explicit google/flan-t5-base model parameter and evaluation i3d features
    !python -W ignore -m scripts.evaluate --checkpoint {best_checkpoint} --data-dir {lm_dir} --i3d-dir {i3d_val_dir} --metadata {metadata_file} --t5-model google/flan-t5-base --num-beams 8 --length-penalty 1.2 --repetition-penalty 1.5
else:
    print("No checkpoints found to evaluate!")

## 8. Qualitative Sample Inference

In [ ]:
# --- Run Qualitative Sample Inference (DRY compliant) ---
import glob
import os
from scripts.inference import run_inference

checkpoint_dirs = sorted(
    glob.glob("results/checkpoints/checkpoint-*"), key=lambda x: int(x.split("-")[-1])
)
if checkpoint_dirs:
    best_checkpoint = checkpoint_dirs[-1]

    # Resolve directories dynamically
    metadata_file = None
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if "realigned_train" in f and f.endswith(".csv"):
                metadata_file = os.path.join(root, f)
                break
        if metadata_file:
            break

    lm_dir = None
    if metadata_file:
        lm_dir = os.path.dirname(os.path.dirname(metadata_file))

    # Resolve Validation I3D Directory
    i3d_val_dir = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if "how2sign-i3d-val" in root.lower() and any(
            f.endswith(".npy") for f in files
        ):
            i3d_val_dir = root
            break

    # Discover landmark filepaths and take first 10 for qualitative checking
    from src.utils.io_utils import discover_landmark_paths

    landmark_files = discover_landmark_paths(lm_dir)
    inference_files = landmark_files[:10]

    if inference_files:
        run_inference(
            checkpoint=best_checkpoint,
            input_paths=inference_files,
            t5_model_name="google/flan-t5-base",
            i3d_dir=i3d_val_dir,
            num_beams=5,
            length_penalty=1.2,
            repetition_penalty=1.5,
        )
    else:
        print("No landmark files found for inference!")
else:
    print("No checkpoints found to run inference!")